In [1]:
# Python 3.10.11
%pip install -r requirements.txt > /dev/null
from args import *

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score

In [3]:
## ## ## ## ## ## ## ## 分割线 ## ## ## ## ## ## ## ##

In [4]:
num_samples = 300  # 样本数量
num_features = 80  # 特征数量

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")  # 检查GPU

expr  = torch.rand(num_samples, num_features)
cnv = torch.rand(num_samples, num_features)
report = torch.rand(num_samples, num_features)
sis = torch.rand(num_samples, num_features)


In [5]:
label = torch.randint(0, 2, (num_samples, 1)).reshape(-1)

In [6]:
# 模型训练批次大小
batchsize = 64

# 定义一些超参数
learning_rate = 0.001
num_epochs = 2

In [7]:
train_dataset = MultiOmicsDataset(expr, cnv, report, sis, label)
train_loader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True, num_workers=3, drop_last=True)

/workspace/pyfaster/examples/model/args.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.expr = torch.tensor(expr, dtype=torch.float32)
/workspace/pyfaster/examples/model/args.py:170: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.cnv = torch.tensor(cnv, dtype=torch.float32)
/workspace/pyfaster/examples/model/args.py:171: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.report = torch.tensor(report, dtype=torch.float32)
/workspace/pyfaster/examples/model/args.py:172: UserWarning: To copy construct from a tensor, it is reco

In [8]:
model = MultiOmicsModel()
# model(expr, cnv, report, sis)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [9]:
def run_fold(fold):    
    for epoch in range(num_epochs):

        # Epoch start

        # Training
        model.train()
        for exper, cnv, report, sis, label in train_loader:
            outputs = model(exper, cnv, report, sis)
            loss = criterion(torch.flatten(outputs), label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # 在训练集上验证模型
        model.eval()  # 设置模型为评估模式
        train_loss = 0
        predictions = [] # 保存预测结果
        true_labels = [] # 保存真实结果
        scores = []      # 保存预测得分
        with torch.no_grad():
            for expr, cnv, report, sis, label in train_loader:
                outputs = model(expr, cnv, report, sis)
                outputs = torch.flatten(outputs)
                train_loss += criterion(outputs, label).item()
                
                predicted = torch.round(torch.sigmoid(outputs))
                predictions.extend(predicted.tolist())
                true_labels.extend(label.tolist())
                scores.extend(outputs.tolist())
        train_loss /= len(train_loader.dataset)
        accuracy = accuracy_score(true_labels, predictions)

        # accuracy_score (真实标签 vs. 预测标签) 计算准确率
        # roc_auc_score  (真实标签 vs. 预测得分) 计算ROC曲线下的面积
        accuracy = accuracy_score(true_labels, predictions)
        auc = roc_auc_score(true_labels, scores)
        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss}, Train Accuracy: {accuracy:.4f}, Train AUC: {auc:.4f}')
        with open(f'{data_output_dir_path}/log_{fold}.txt', 'a') as FO:
            FO.writelines(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss}, Train Accuracy: {accuracy:.4f}, Train AUC: {auc:.4f}\n')
        
        # 在测试集上验证模型
        model.eval()  # 设置模型为评估模式
        test_loss = 0
        predictions = []
        true_labels = []
        scores = []
        with torch.no_grad():
            for expr, cnv, report, sis, label in train_loader:
                outputs = model(expr, cnv, report, sis)
                outputs = torch.flatten(outputs)
                test_loss += criterion(outputs, label).item()
                predicted = torch.round(torch.sigmoid(outputs)) 
                predictions.extend(predicted.tolist())
                true_labels.extend(label.tolist())
                scores.extend(outputs.tolist())
        # 打印训练过程中的损失和测试精度
        test_loss /= len(train_loader.dataset)
        accuracy = accuracy_score(true_labels, predictions)
        auc = roc_auc_score(true_labels, scores)
        print(f'Epoch [{epoch+1}/{num_epochs}], Test Loss: {test_loss}, Test Accuracy: {accuracy:.4f}, Test AUC: {auc:.4f}\n')
        with open(f'{data_output_dir_path}/log_{fold}.txt', 'a') as FO:
            FO.writelines(f'Epoch [{epoch+1}/{num_epochs}], Test Loss: {test_loss}, Test Accuracy: {accuracy:.4f}, Test AUC: {auc:.4f}\n')


In [11]:
# 实例化五折交叉验证对象
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
fold = 0  # 折数
for train_idx, test_idx in cv.split(expr, label):
    print(f'fold: {fold}')
    fold += 1
    print(f"Train index length: {len(train_idx)}")
    print(f"Test index length: {len(test_idx)}")
    run_fold(fold)
    

fold: 0
Train index length: 150
Test index length: 150
Epoch [1/2], Train Loss: 0.009122740427652995, Train Accuracy: 0.5742, Train AUC: 0.5197
Epoch [1/2], Test Loss: 0.009096508026123046, Test Accuracy: 0.5859, Test AUC: 0.5435

Epoch [2/2], Train Loss: 0.009114933609962463, Train Accuracy: 0.5742, Train AUC: 0.5339
Epoch [2/2], Test Loss: 0.00914420207341512, Test Accuracy: 0.5625, Test AUC: 0.5658

fold: 1
Train index length: 150
Test index length: 150
Epoch [1/2], Train Loss: 0.009104552666346232, Train Accuracy: 0.5781, Train AUC: 0.5125
Epoch [1/2], Test Loss: 0.00915372888247172, Test Accuracy: 0.5586, Test AUC: 0.5652

Epoch [2/2], Train Loss: 0.009125432968139648, Train Accuracy: 0.5664, Train AUC: 0.4629
Epoch [2/2], Test Loss: 0.009140831430753072, Test Accuracy: 0.5625, Test AUC: 0.5321

